# Introduction to dbt (Data Build Tool)

Now that we have successfully indexed blockchain transfer events and saved them to our PostgreSQL database using Envio, we need to transform this raw data into clean, business-ready models.

In this lesson, we will introduce **dbt (data build tool)**, set up a dbt project, and write our first data transformation models.

---

## 1. What is dbt and Why Use It?

In modern data warehousing, we follow the **ELT** (Extract, Load, Transform) framework. We extract raw data from the blockchain, load it directly into our PostgreSQL warehouse, and *then* transform it using the database's own processing engine.

**dbt** is the industry-standard tool for managing these SQL transformations. Instead of writing messy, custom scripts to execute SQL queries, dbt allows us to:

* Write transformations in clean, modular SQL `SELECT` statements.
* Automatically compile models and manage table dependencies (dbt knows which tables need to be built first).
* Perform automated data quality tests (e.g. testing for nulls or duplicates).


## 2. Installing dbt

dbt is a Python-based library. The easiest way to install it alongside our Postgres database connector is using `uv`:

```bash
uv add dbt-postgres
```

To verify it installed correctly, check the version:
```bash
uv run dbt --version
```

## 3. Initializing a dbt Project


Create a new directory and navigate to it
```bash
mkdir dbt-transform && cd dbt-transform
```

initialize dbt workspace:

```bash
uv run dbt init dbt_warehouse
```

The wizard will ask you which database connector you are using. Select **`postgres`**.

follow the prompt and enter you postgres credentials

### Project Structure
A standard dbt project folder contains:
* **`dbt_project.yml`**: The main configuration file defining the project name and how models are built.
* **`models/`**: The directory where we write our SQL transformation files.
* **`seeds/`**: Used for loading static CSV data files into the database.

## 4. Connecting dbt to our Postgres Warehouse

dbt keeps database connection credentials separate from the project code inside a global file named `profiles.yml` (usually located in your home directory at `~/.dbt/profiles.yml`). You can also create this file in the dbt project directory

Configure your profile to connect to your local Docker Postgres container:
Note: If you followed the setup prompt and entered your credential, then it should appear as below too

```bash
dbt_warehouse:
  outputs:
    dev:
      type: postgres
      host: localhost
      port: 5432
      user: root
      password: root
      dbname: data-set
      schema: analytics
      threads: 1
  target: dev
```

To test that dbt can successfully talk to your Postgres container, run:
```bash
uv run dbt debug
```

The output should end with `All checks passed!`

## 5. Declaring Raw Data Sources

Before we write staging models, we must define our **Sources**. In dbt, a source is a declaration of where our raw, un-transformed tables live in our database. Defining sources allows dbt to build lineage graphs and document raw data schemas.

Create a new schema file named `models/staging/sources.yml`:

```yaml
version: 2

sources:
  - name: dataset           # The name of the source database group
    schema: analytics          # The Postgres schema where Envio inserts raw tables
    tables:
      - name: Transfer     # The exact table name generated by the indexer
```

Once defined, we can reference this raw table inside our SQL queries using the syntax `{{ source('analytics', 'Transfer') }}`. This is much better than hardcoding `analytics.Transfer` directly, because it allows dbt to dynamically adapt if schema names change.

## 6. Staging and Naming Conventions

The first step in any dbt pipeline is building the **Staging Layer**.

### Staging Location
Staging models should always be organized inside their own subdirectory, usually at `models/staging/`.

### The `stg_` Naming Convention
In professional analytics engineering, staging models are prefixed with **`stg_`** (e.g., `stg_transfers.sql`).
* **Is this prefix mandatory?** No. dbt will compile and run your model perfectly fine under any filename (e.g. `clean_transfers.sql`).
* **Why do we use it?** It is an industry-standard best practice. Prefixes make it immediately obvious to other engineers which files are simple cleaning scripts (`stg_`), which are intermediate joins (`int_`), and which are final business metrics (`mart_`).

Create your staging file at `models/staging/stg_transfers.sql`:

```sql
with raw_transfers as (
    select * from {{ source('dataset', 'Transfer') }}
)

select
    id as transfer_id,
    to_timestamp("blockTimestamp") as block_time,
    lower("from") as sender_address,
    lower("to") as recipient_address,
    value as raw_value,
    "blockNumber" as block_number,
    txhash,
    "tokenAddress"
from raw_transfers
where value > 0
```

## 7. The Marts Layer (`mart_transfer_volume.sql`)

Mart models aggregate data for business consumption. Here, we calculate the total tokens transferred by each address. Create the mart file at `models/marts/mart_transfer_volume.sql`:

```sql
with transfers as (
    select * from {{ ref('stg_transfers') }}
)

select
    sender_address as user_address,
    'sent' as transaction_type,
    sum(raw_value) as total_raw_value,
    count(*) as total_transfers
from transfers
group by sender_address

union all

select
    recipient_address as user_address,
    'received' as transaction_type,
    sum(raw_value) as total_raw_value,
    count(*) as total_transfers
from transfers
group by recipient_address
```

## 8. Running and Validating dbt Models

Once your SQL models are created, instruct dbt to execute the transformations and build the tables inside Postgres:

```bash
uv run dbt run
```

dbt will compile the SQL, execute it against your Postgres instance, and create new tables inside the `envio` schema. You can view these tables inside PGAdmin, DBeaver, or directly query them to start analyzing wallet patterns!

## 9. Running Specific Models (`--select` / `-s`)

In a production data warehouse, you might have hundreds of models. Rebuilding the entire database every time you make a minor edit is highly inefficient.

To tell dbt to compile and execute **only one specific model** (and skip all others), use the `--select` (or shorthand `-s`) flag followed by the model filename:

```bash
# Run only the staging transfers model
uv run dbt run --select stg_transfers
```

This compile strategy speeds up development cycles drastically by updating only target tables.

## 10. Data Quality Auditing with dbt Tests

dbt allows us to write assertions (tests) about our data directly inside configuration YAML files. dbt comes pre-packaged with four standard tests:
* `unique`: Verifies that values in a column do not contain duplicates.
* `not_null`: Ensures that no rows contain empty values.
* `accepted_values`: Validates that column data falls within a strict list.
* `relationships`: Validates foreign key integrity between two models.

Create a new schema file named `models/staging/schema.yml` to define data quality validations on our staging table:

```yaml
version: 2

models:
  - name: stg_transfers
    description: "Cleaned token transfers table"
    columns:
      - name: transfer_id
        tests:
          - unique
          - not_null
      - name: sender_address
        tests:
          - not_null
```

To execute your validation tests against the active tables, run:
```bash
uv run dbt test
```

## 11. Customizing Target Schemas & Table Aliases

Sometimes, dbt's default behavior needs to be overridden. For example:
* You want to save a table under a **custom schema namespace** in Postgres (e.g. putting raw models in a `staging` schema and business tables in a `marts` schema).
* You want to assign a **custom name (alias)** to the generated table in the database that is different from its SQL filename in dbt.

We can configure these settings dynamically by writing a `config` block at the very top of our SQL models.

### A. Configuring Custom Schemas
Add this configuration to the top of your model file:
```sql
{{
    config(
        materialized = 'table',
        schema = 'staging'
    )
}}
```
When you run this model, dbt compiles it and creates the table in Postgres under the custom namespace schema `envio_staging` (dbt automatically appends custom schemas to the default profile schema for safety).

### B. Renaming Database Tables with Aliases
By default, if your SQL script is named `stg_transfers.sql`, dbt creates a table named `stg_transfers` in the database. 

If you want the database table to have a cleaner name, add the `alias` property:
```sql
{{
    config(
        materialized = 'table',
        schema = 'staging',
        alias = 'clean_transfers'
    )
}}
```

**Before vs. After Comparison:**
* **Without config**: Reference name in dbt is `stg_transfers`, and the compiled table in Postgres is named **`stg_transfers`**.
* **With config**: Reference name in dbt is still `stg_transfers` (for code references), but inside the Postgres database, the physical table is saved as **`clean_transfers`**.